# Phase 5: Auswertung & SQL-Bonus

**Ziel:** Modellleistung richtig bewerten, Ergebnisse in SQLite speichern und abfragen.

**StackFuel-Themen:** Statistik · SQL Basics (SELECT, WHERE, GROUP BY, JOIN)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score
)

X_train = np.load('../data/processed/X_train.npy')
X_val   = np.load('../data/processed/X_val.npy')
y_train = np.load('../data/processed/y_train.npy')
y_val   = np.load('../data/processed/y_val.npy')
classes = np.load('../data/processed/classes.npy')

# Bestes Modell nochmal trainieren (oder aus Notebook 03 übernehmen)
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_val)

print('Modell bereit.')

## 5.1 Klassifikationsbericht

In [ ]:
print(classification_report(y_val, y_pred, target_names=classes))

# Warum nicht nur Accuracy?
# Bei unausgeglichenen Klassen (Imbalance) ist Accuracy irreführend.
# Precision: Wie viele der als 'anomal' klassifizierten Fälle sind wirklich anomal?
# Recall:    Wie viele der echten Anomalien wurden erkannt?
# F1-Score:  Harmonisches Mittel aus Precision und Recall – gut bei Imbalance.

## 5.2 Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_val, y_pred,
    display_labels=classes,
    ax=ax,
    colorbar=False,
    cmap='Blues'
)
ax.set_title('Confusion Matrix – Validation Set')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 5.3 SQL-Bonus: Ergebnisse in SQLite speichern

`sqlite3` ist in Python eingebaut – kein extra Package nötig!

In [ ]:
# Vorhersagen als DataFrame
df_preds = pd.DataFrame({
    'sample_id': range(len(y_val)),
    'true_label':  [classes[i] for i in y_val],
    'pred_label':  [classes[i] for i in y_pred],
    'correct':     (y_val == y_pred).astype(int)
})

# Modell-Metadaten
df_models = pd.DataFrame([{
    'model_id': 1,
    'model_name': 'RandomForest',
    'n_estimators': 100,
    'accuracy': accuracy_score(y_val, y_pred).round(4),
    'f1_macro': f1_score(y_val, y_pred, average='macro').round(4)
}])

# SQLite-Datenbank erstellen
conn = sqlite3.connect('../data/results.db')

df_preds.to_sql('predictions', conn, if_exists='replace', index=False)
df_models.to_sql('models', conn, if_exists='replace', index=False)

print('Tabellen gespeichert: predictions, models')

## 5.4 SQL-Abfragen üben

In [ ]:
# SELECT + WHERE
query1 = "SELECT * FROM predictions WHERE correct = 0 LIMIT 10"
df_errors = pd.read_sql(query1, conn)
print('Fehlklassifikationen (erste 10):')
print(df_errors)

In [ ]:
# GROUP BY – Genauigkeit pro Klasse
query2 = """
SELECT 
    true_label,
    COUNT(*) AS gesamt,
    SUM(correct) AS korrekt,
    ROUND(AVG(correct) * 100, 1) AS accuracy_pct
FROM predictions
GROUP BY true_label
ORDER BY accuracy_pct DESC
"""
df_by_class = pd.read_sql(query2, conn)
print('Accuracy pro Klasse:')
print(df_by_class.to_string(index=False))

In [ ]:
# JOIN – Vorhersagen mit Modellinformationen verknüpfen
query3 = """
SELECT p.true_label, p.pred_label, p.correct, m.model_name, m.accuracy
FROM predictions p
JOIN models m ON m.model_id = 1
WHERE p.correct = 0
LIMIT 5
"""
df_joined = pd.read_sql(query3, conn)
print('JOIN-Beispiel:')
print(df_joined.to_string(index=False))

conn.close()

## 5.5 Abschlusszusammenfassung

_Hier deine eigene Reflexion:_

- **Bestes Modell & Score:** ...
- **Schwierigste Klasse (laut Confusion Matrix):** ...
- **Was ich beim nächsten Projekt anders machen würde:** ...
- **Was ich für das StackFuel-Training mitgenommen habe:** ...